# Baseline — MLP résiduel actuel

**Objectif :** Reproduire le modèle de production actuel (TSLClassifier) sur le protocole propre (sans fuite train/test) pour servir de baseline.

Le modèle actuel est un MLP résiduel profond (126 → 512 → 512 → 256 → 256 → 128 → 64 → 57)
avec BatchNorm, ReLU, Dropout 0.2, init Kaiming. Il consomme les 126 features normalisées
(2 mains × 21 landmarks × 3 coordonnées, relatives au poignet et normalisées par la taille de la main).

**Protocole expérimental (commun à toutes les expériences) :**
- Dataset : 7 664 images du dataset W7CH (TSL), landmarks MediaPipe (21 points × 2 mains).
- Split stratifié 80/10/10 (seed 42) sur les échantillons ORIGINAUX, **avant**
  toute augmentation → aucune fuite de données entre train et validation/test.
- Augmentation (rotation ±20°, échelle 0.85-1.15, translation ±0.1, bruit 0.02)
  appliquée **uniquement** sur le train, équilibré à 200 échantillons/classe.
- Entraînement : Adam (lr 1e-3, wd 1e-4), CrossEntropy + label smoothing 0.1,
  ReduceLROnPlateau, early stopping sur val_acc, seeds fixes.
- 56 classes effectives sur 57 (les images `metro` ne sont pas détectées par
  MediaPipe), tête de classification à 57 sorties (comme le modèle actuel).


In [1]:
import sys
from pathlib import Path

NB_DIR = Path.cwd()
ROOT = NB_DIR
while not (ROOT / "common").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
for p in [str(ROOT), str(ROOT / "common")]:
    if p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader

from common.paths import RESULTS_DIR, MODELS_DIR, PLOTS_DIR, REPO_ROOT
from common.data import load_dataset, normalized_inputs
from common.models import (
    count_params, set_seed, BaselineMLP, LandmarkTransformer, HandGCN, HandGAT,
)
from common.train import (
    Trainer, predict, full_metrics, save_metrics,
    plot_training_curves, plot_confusion, run_torch_benchmark,
)
from common import export as ex
from common.robustness import robustness_summary

set_seed(42)

def load_class_names():
    with open(REPO_ROOT / "data" / "processed" / "class_names.txt", encoding="utf-8") as f:
        return [l.strip() for l in f if l.strip()]

class_names = load_class_names()
print("torch", torch.__version__)


torch 2.13.0+cpu


## 1. Chargement des données

Le dataset préparé (split + augmentation, cf. README) est chargé. Le modèle
reçoit vecteur normalisé de 126 features (2 × 21 × 3, relatif poignet + taille main).


In [2]:
dataset = load_dataset()
inp = normalized_inputs(dataset, with_geo=False)
X_train, y_train = torch.from_numpy(inp["train"]["features"]).float(), torch.from_numpy(inp["train"]["labels"])
X_val, y_val     = torch.from_numpy(inp["val"]["features"]).float(),   torch.from_numpy(inp["val"]["labels"])
X_test, y_test   = torch.from_numpy(inp["test"]["features"]).float(),  torch.from_numpy(inp["test"]["labels"])
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=128, shuffle=True)
val_loader   = DataLoader(TensorDataset(X_val, y_val), batch_size=256, shuffle=False)
test_loader  = DataLoader(TensorDataset(X_test, y_test), batch_size=256, shuffle=False)
print("train:", tuple(X_train.shape), "| val:", tuple(X_val.shape), "| test:", tuple(X_test.shape))


train: (13282, 126) | val: (767, 126) | test: (767, 126)


## 2. Architecture

MLP résiduel de production (src/models/classifier.py::TSLClassifier) :
input 126 → 512 (+ skip 512) → 256 (+ skip 256) → 128 → 64 → 57.
BatchNorm + ReLU + Dropout(0.2) à chaque bloc, label smoothing 0.1.

Nombre de paramètres : mesuré ci-dessous.


In [3]:
model = BaselineMLP(num_classes=57, input_size=126)
print(f"params: {count_params(model):,}")


params: 903,097


## 3. Entraînement

Mêmes hyperparamètres que le protocole commun (Adam lr 1e-3, wd 1e-4,
label smoothing 0.1, early stopping patience 20, max 120 epochs).


In [4]:
trainer = Trainer(model, patience=20)
history = trainer.fit(train_loader, val_loader, max_epochs=120, early_stop=True)
trainer.restore_best()

plot_training_curves(history, PLOTS_DIR / "training_curves" / "baseline_mlp_curves.png",
                     title="Baseline — MLP résiduel actuel")
print("best val acc:", trainer.best_val_acc)


  ep 1/120 train 3.6697/0.1479 val 2.5350/0.4289 (1.5s) best 0.4289


  ep 2/120 train 2.8336/0.3344 val 2.0658/0.5828 (1.0s) best 0.5828


  ep 3/120 train 2.4399/0.4417 val 1.7937/0.6688 (1.5s) best 0.6688


  ep 4/120 train 2.1878/0.5184 val 1.6169/0.7001 (1.0s) best 0.7001


  ep 5/120 train 2.0138/0.5700 val 1.5328/0.7080 (1.5s) best 0.7080


...

  ep 117/120 train 0.7945/0.9913 val 0.8110/0.9765 (1.5s) best 0.9804


  ep 118/120 train 0.7935/0.9914 val 0.8049/0.9791 (1.0s) best 0.9804


  ep 119/120 train 0.7930/0.9920 val 0.8097/0.9791 (1.5s) best 0.9804


  ep 120/120 train 0.7956/0.9901 val 0.8045/0.9765 (1.0s) best 0.9804


best val acc: 0.9804432855280313


## 4. Évaluation sur le jeu de test

Métriques complètes : accuracy, precision/recall/F1 (macro et pondérés),
matrice de confusion, performance par classe.


In [5]:
y_pred, y_true = predict(model, test_loader)
metrics = full_metrics(y_true, y_pred, num_classes=57)
print(f"test accuracy: {metrics['accuracy']:.4f}")
print(f"macro F1: {metrics['macro_f1']:.4f} | weighted F1: {metrics['f1']:.4f}")
print(f"macro precision: {metrics['macro_precision']:.4f} | macro recall: {metrics['macro_recall']:.4f}")

cm = np.array(metrics["confusion_matrix"])
plot_confusion(cm, class_names, PLOTS_DIR / "confusion_matrices" / "baseline_mlp_cm.png",
               title="Baseline — MLP résiduel actuel — confusion matrix (test)")
np.save(RESULTS_DIR / "confusion_baseline_mlp.npy", cm)

import pandas as pd
per = pd.DataFrame({
    "class": class_names,
    "precision": metrics["per_class_precision"],
    "recall": metrics["per_class_recall"],
    "f1": metrics["per_class_f1"],
    "support": metrics["per_class_support"],
})
per.to_csv(RESULTS_DIR / "per_class_baseline_mlp.csv", index=False)
print(per.sort_values("f1").head(8))


test accuracy: 0.9817
macro F1: 0.9802 | weighted F1: 0.9818
macro precision: 0.9827 | macro recall: 0.9807


     class  precision    recall        f1  support
54   metro   0.000000  0.000000  0.000000        0
10  mar7ba   0.750000  1.000000  0.857143        3
8    labes   1.000000  0.750000  0.857143        4
44    5mis   0.785714  1.000000  0.880000       11
5       cv   1.000000  0.818182  0.900000       11
22  telvza   1.000000  0.833333  0.909091        6
13     non   0.866667  1.000000  0.928571       13
39   jadda   0.888889  1.000000  0.941176       16


## 5. Robustesse

Le modèle est évalué sous perturbations des landmarks appliquées **avant**
normalisation (dans l'espace image) : bruit gaussien, rotation, translation,
changement d'échelle — 5 essais par perturbation, moyenne ± écart-type.


In [6]:
from common.data import wrist_normalize, flatten
def forward_fn(model, raw, batch_size=512):
    model.eval()
    feats = flatten(wrist_normalize(raw))
    outs = []
    with torch.no_grad():
        for i in range(0, len(feats), batch_size):
            x = torch.from_numpy(feats[i:i + batch_size]).float()
            outs.append(model(x).argmax(1).numpy())
    return np.concatenate(outs)
rob = robustness_summary(model, dataset["test"]["img_landmarks"],
                         dataset["test"]["labels"], base_acc=metrics["accuracy"],
                         forward_fn=forward_fn)


  noise_0.01: acc=0.8300 (drop=+0.1518)
  noise_0.02: acc=0.7867 (drop=+0.1950)
  noise_0.05: acc=0.5622 (drop=+0.4196)
  rotation_5: acc=0.9726 (drop=+0.0091)
  rotation_10: acc=0.9361 (drop=+0.0456)


  rotation_20: acc=0.8566 (drop=+0.1252)


  translation_0.05: acc=0.9817 (drop=+0.0000)
  translation_0.10: acc=0.9817 (drop=+0.0000)
  scale_0.90: acc=0.9817 (drop=+0.0000)
  scale_1.10: acc=0.9817 (drop=+0.0000)


## 6. Mesures mobiles

- Nombre de paramètres et taille du checkpoint PyTorch.
- Export ONNX (FP32), vérification, latence ONNX Runtime CPU (batch=1).
- Versions FP16 et INT8 (quantification dynamique) : taille, latence,
  dégradation d'accuracy sur le jeu de test.


In [7]:
sample = X_test[:1].float()
lat_torch = run_torch_benchmark(model, sample)
print(f"torch latency: {lat_torch:.3f} ms/sample")

onnx_path = ex.export_onnx(model, sample, "baseline_mlp")
fp16 = ex.quantize_fp16(onnx_path, "baseline_mlp")
int8 = ex.quantize_int8(onnx_path, "baseline_mlp")

quantized = {}
for tag, p in [("fp32", onnx_path), ("fp16", fp16), ("int8", int8)]:
    preds = ex.ort_predict(p, X_test.numpy())
    quantized[tag] = {
        "size_mb": ex.file_size_mb(p),
        "latency_ms": ex.ort_latency(p, sample.numpy()),
        "test_acc": float((preds == y_test.numpy()).mean()),
    }
    print(f"  {tag}: size={quantized[tag]['size_mb']:.2f} MB "
          f"latency={quantized[tag]['latency_ms']:.3f} ms "
          f"acc={quantized[tag]['test_acc']:.4f}")

extra = {
    "robustness": rob,
    "epochs": trainer.epochs_run,
    "params": count_params(model),
    "best_val_acc": trainer.best_val_acc,
    "torch_latency_ms": lat_torch,
    "quantized": quantized,
}
save_metrics("baseline_mlp", metrics, extra=extra)

torch_path = ex.save_torch(model, "baseline_mlp")
print("saved:", torch_path, f"({ex.file_size_mb(torch_path):.2f} MB)")


torch latency: 0.303 ms/sample

  fp32: size=3.47 MB latency=0.092 ms acc=0.9817
  fp16: size=1.74 MB latency=0.095 ms acc=0.9817
  int8: size=0.92 MB latency=0.091 ms acc=0.9831
metrics saved -> D:\Stage\Tunisian-Sign-Language-Translator\alternative_models\results\metrics_baseline_mlp.json
saved: D:\Stage\Tunisian-Sign-Language-Translator\alternative_models\models\baseline_mlp.pth (3.48 MB)


## 7. Conclusions

Le MLP actuel sert de référence. On mesure ici sa performance réelle sur un protocole
sans fuite de données (le protocole historique appliquait l'augmentation avant le split,
ce qui pouvait faire apparaître des copies augmentées du même échantillon dans train ET test).
Toute amélioration des architectures alternatives devra être mesurée relativement à cette baseline.
